# 第9章 监督学习基础 —— 配套代码

对应正文 `book/part3/09-supervised-learning.md`。

> **运行前准备**：请先在终端执行 `uv run python scripts/make_sample_data.py` 生成内置示例数据。

## 演示内容

1. 环境初始化与数据加载
2. 构造滑滞特征（延迟收益、动量、波动率）
3. 构造分类标签（明日涨跌）并验证无前视偏差
4. 数据集描述性统计
5. 随机划分 vs 时序划分：**展示前视偏差导致准确率虚高**
6. TimeSeriesSplit 交叉验证 + 逻辑回归
7. 岖回归（L2正则化）与正则化路径图
8. 评估指标：ROC/AUC 图 + 信息系数 IC/RankIC
9. 展示：完整 Pipeline（标准化 + 模型）防止数据泄露
10. 习题参考解答

In [ ]:
# === 在线运行引导（Google Colab）。本地 / Binder 会自动跳过 ===
import sys

if "google.colab" in sys.modules:
    import os
    import subprocess

    if not os.path.isdir("src/fds"):  # 尚未进入仓库
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/albertandking/financial-data-science", "fds_book"],
                       check=True)
        os.chdir("fds_book")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
        print("已就绪：仓库已克隆、fds 已安装、内置数据随仓库一并克隆。")

In [ ]:
# Cell 1: 环境初始化与数据加载
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from scipy.stats import pearsonr, spearmanr

from sklearn.linear_model import LogisticRegression, Ridge, Lasso, RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import (
    TimeSeriesSplit, KFold, cross_val_score, train_test_split
)
from sklearn.metrics import (
    accuracy_score, classification_report, roc_auc_score,
    roc_curve, mean_squared_error, r2_score
)

from fds import load_sample_prices, load_market, daily_returns, set_chinese_font

set_chinese_font()
np.random.seed(42)

# 加载数据
prices = load_sample_prices()
market = load_market()
rets   = daily_returns(prices)     # 日简单收益率

TICKERS = list(prices.columns)    # ['BANK', 'LIQUOR', 'TECH', 'UTILITY']

print(f'价格数据维度：{prices.shape}')
print(f'收益率数据维度：{rets.shape}')
print(f'日期范围：{rets.index[0].date()} 至 {rets.index[-1].date()}')
print(f'股票列：{TICKERS}')
rets.head(3)

## 9.2 构造监督学习特征与标签

我们以 **TECH** 股票为示例，构造以下**技术类特征**（均已错开一期，无前视偏差）：

| 特征名 | 构造方式 | 金融含义 |
|--------|---------|------|
| `ret_1` | `shift(1)` 昨日收益 | 短期反转/动量 |
| `ret_5` | `rolling(5).sum().shift(1)` | 周级别动量 |
| `ret_20` | `rolling(20).sum().shift(1)` | 月级别动量 |
| `vol_20` | `rolling(20).std().shift(1)` | 波动率异质 |
| `ma_signal` | `(MA5/MA20 - 1).shift(1)` | 均线偿离 |
| `mkt_ret_5` | 市场过去5日收益 | 市场状态 |

**标签**：明日涨跌方向（二分类：1=涨，0=跌）

$$y_t = \mathbf{1}[r_{t+1} > 0]$$

> **重要**：`shift(-1)` 取明日收益作为标签；所有特征用 `shift(1)` 或更大偏移确保不前视。

In [ ]:
# Cell 2: 构造特征与标签

TICKER = 'TECH'   # 以 TECH 为主要示例

r = rets[TICKER].copy()           # TECH 日收益率
mkt_r = market['index_return']    # 市场指数日收益

# 构造特征（所有特征均 shift 到 t-1 时刻已知）
feat = pd.DataFrame(index=r.index)
feat['ret_1']    = r.shift(1)                           # 昨日收益
feat['ret_5']    = r.rolling(5).sum().shift(1)          # 过去5日收益
feat['ret_20']   = r.rolling(20).sum().shift(1)         # 过去20日收益
feat['vol_20']   = r.rolling(20).std().shift(1)         # 过去20日波动率

# 均线相对偏离：(5日均线 / 20日均线 - 1)
ma5  = prices[TICKER].rolling(5).mean()
ma20 = prices[TICKER].rolling(20).mean()
feat['ma_signal'] = (ma5 / ma20 - 1).shift(1)

feat['mkt_ret_5'] = mkt_r.rolling(5).sum().shift(1)    # 市场5日收益

# 标签：明日涨跌（shift(-1)取明日实际收益）
label = (r.shift(-1) > 0).astype(int)
label.name = 'up_tomorrow'

# 对齐：删除 NaN（因 rolling 产生）
data = feat.join(label).dropna()
X_all = data[feat.columns]
y_all = data['up_tomorrow']

print(f'特征数：{X_all.shape[1]}，样本量：{X_all.shape[0]}')
print(f'标签分布：y=1(涨)占比: {y_all.mean():.1%}，y=0(跌)占比: {1-y_all.mean():.1%}')
print(f'\n最早日期：{data.index[0].date()}，最晚日期：{data.index[-1].date()}')
print('\n特征前5行：')
X_all.head()

## 9.3 特征与标签描述性统计

在建模前，先直观了解数据的基本属性。

In [ ]:
# Cell 3: 描述性统计与延迟收益特征相关性

print('=== 特征描述性统计 ===')
print(X_all.describe().round(4).to_string())

print('\n=== 特征与明日涨跌的 Pearson 相关系数 ===')
corrs = {}
for col in X_all.columns:
    r_val, p_val = pearsonr(X_all[col], y_all)
    corrs[col] = {'Pearson r': round(r_val, 4), 'p-value': round(p_val, 4)}
df_corr = pd.DataFrame(corrs).T
df_corr['significant'] = df_corr['p-value'] < 0.05
print(df_corr.to_string())

# 延迟收益 vs 明日涨跌可视化
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, feat_name in zip(axes, ['ret_1', 'ret_5', 'ret_20']):
    up   = X_all.loc[y_all==1, feat_name]
    down = X_all.loc[y_all==0, feat_name]
    ax.hist(down, bins=30, alpha=0.6, color='crimson',   label='跌天样本', density=True)
    ax.hist(up,   bins=30, alpha=0.6, color='steelblue', label='涨天样本', density=True)
    ax.set_title(f'{feat_name} 分布（不同标签）', fontsize=11)
    ax.set_xlabel(feat_name)
    ax.set_ylabel('概率密度')
    ax.legend(fontsize=9)

plt.suptitle('TECH: 延迟收益特征在涨跌天的分布对比', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 9.4 前视偏差实验：随机划分 vs 时序划分

**核心问题**：如果我们错误地使用随机K折交叉验证，模型准确率会虚高多少？

**实验设计**：
- 随机划分（错误）：`train_test_split` 随机打乱
- 时序划分（正确）：前70%为训练集，后30%为测试集
- 两种方式均使用相同的 Pipeline（StandardScaler + LogisticRegression）

In [ ]:
# Cell 4: 前视偏差实验：随机划分 vs 时序划分

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    LogisticRegression(C=0.1, max_iter=1000, random_state=42))
])

# --- 方式 A: 随机划分（错误！）---
X_tr_rnd, X_te_rnd, y_tr_rnd, y_te_rnd = train_test_split(
    X_all, y_all, test_size=0.3, random_state=42
)
pipe.fit(X_tr_rnd, y_tr_rnd)
acc_rnd = accuracy_score(y_te_rnd, pipe.predict(X_te_rnd))

# --- 方式 B: 时序划分（正确）---
split_idx = int(len(X_all) * 0.7)
X_tr_ts, X_te_ts = X_all.iloc[:split_idx], X_all.iloc[split_idx:]
y_tr_ts, y_te_ts = y_all.iloc[:split_idx], y_all.iloc[split_idx:]

pipe.fit(X_tr_ts, y_tr_ts)
acc_ts = accuracy_score(y_te_ts, pipe.predict(X_te_ts))

# --- 基准线：绝对多数类局算法 ---
acc_baseline = max(y_all.mean(), 1 - y_all.mean())

print('=' * 50)
print('前视偏差实验结果')
print('=' * 50)
print(f'随机划分准确率  : {acc_rnd:.4f}  \u2190 这是假的！包含前视偏差')
print(f'时序划分准确率  : {acc_ts:.4f}  \u2190 这才是真实的样外性能')
print(f'基准线（多数类）: {acc_baseline:.4f}')
print(f'\n虚高幅度 (随机 - 时序): +{acc_rnd - acc_ts:.4f} '
      f'({(acc_rnd - acc_ts)*100:.1f}个百分点)')

print('\n训练集日期范围（时序）：'
      f'{X_tr_ts.index[0].date()} \u2192 {X_tr_ts.index[-1].date()}')
print('测试集日期范围（时序）：'
      f'{X_te_ts.index[0].date()} \u2192 {X_te_ts.index[-1].date()}')

## 9.5 TimeSeriesSplit 交叉验证

`TimeSeriesSplit` 保证训练集总在验证集之前，逐步■扩展训练窗口：

```
Fold 1: 训练[===...] 验证[---]
Fold 2: 训练[=======...] 验证[---]
Fold 3: 训练[===========...] 验证[---]
```

下面同时展示 `KFold`（随机）与 `TimeSeriesSplit` 的平均准确率差距。

In [ ]:
# Cell 5: KFold vs TimeSeriesSplit 交叉验证对比

pipe_cv = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    LogisticRegression(C=0.1, max_iter=1000, random_state=42))
])

# KFold（随机打乱，错误）
kf   = KFold(n_splits=5, shuffle=True, random_state=42)
scores_kf = cross_val_score(pipe_cv, X_all, y_all, cv=kf, scoring='accuracy')

# TimeSeriesSplit（时序，正确）
tscv  = TimeSeriesSplit(n_splits=5)
scores_ts = cross_val_score(pipe_cv, X_all, y_all, cv=tscv, scoring='accuracy')

print('CV 方式对比（单股票 TECH）')
print('=' * 55)
print(f'KFold(随机)   : {scores_kf.mean():.4f} \u00b1 {scores_kf.std():.4f}')
print(f'  每折得分: {[round(s,4) for s in scores_kf]}')
print(f'TimeSeriesSplit: {scores_ts.mean():.4f} \u00b1 {scores_ts.std():.4f}')
print(f'  每折得分: {[round(s,4) for s in scores_ts]}')
print(f'\n虚高幅度: +{scores_kf.mean() - scores_ts.mean():.4f}')
print('结论: KFold 的高分源于前视偏差，在实盘中无法复现。')

# 可视化: 每折得分
fig, ax = plt.subplots(figsize=(9, 5))
folds = list(range(1, 6))
ax.plot(folds, scores_kf, 'o-', color='crimson',   lw=2, markersize=8, label=f'KFold (随机打乱) 均={scores_kf.mean():.3f}')
ax.plot(folds, scores_ts, 's-', color='steelblue', lw=2, markersize=8, label=f'TimeSeriesSplit 均={scores_ts.mean():.3f}')
ax.axhline(acc_baseline, color='gray', lw=1.5, linestyle='--', label=f'基准线={acc_baseline:.3f}')
ax.set_xlabel('交叉验证折数')
ax.set_ylabel('准确率')
ax.set_title('KFold vs TimeSeriesSplit: 每折准确率对比\n（差距 = 前视偏差导致的虚高）', fontsize=12)
ax.legend(fontsize=10)
ax.set_xticks(folds)
ax.set_ylim(0.4, 0.75)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 9.6 岖回归（L2正则化）与正则化路径

岖回归将损失函数修改为：

$$\min_{\boldsymbol{\beta}} \left\{ \|y - X\boldsymbol{\beta}\|_2^2 + \lambda \|\boldsymbol{\beta}\|_2^2 \right\}$$

随 $\lambda$ 增大，所有系数均向零收缩。通过绘制正则化路径图，可展示各特征系数随正则化强度的变化。

In [ ]:
# Cell 6: 岖回归正则化路径图 + RidgeCV 选参

# 标准化特征（在训练集上 fit）
scaler_demo = StandardScaler()
X_tr_scaled = scaler_demo.fit_transform(X_tr_ts)
X_te_scaled = scaler_demo.transform(X_te_ts)

# 正则化路径
alphas = np.logspace(-3, 3, 60)
coef_path = []
for a in alphas:
    ridge = Ridge(alpha=a)
    ridge.fit(X_tr_scaled, y_tr_ts.astype(float))
    coef_path.append(ridge.coef_)

coef_path = np.array(coef_path)   # shape: (60, n_features)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# 左：正则化路径图
ax1 = axes[0]
colors_path = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']
for i, col in enumerate(X_all.columns):
    ax1.plot(np.log10(alphas), coef_path[:, i],
             color=colors_path[i], lw=2, label=col)
ax1.axhline(0, color='black', lw=0.8, linestyle=':')
ax1.set_xlabel('log10(lambda)')
ax1.set_ylabel('系数大小')
ax1.set_title('岖回归正则化路径\n(lambda增大\u2192各系数屁缩向零)', fontsize=11)
ax1.legend(fontsize=9, loc='upper right')
ax1.grid(alpha=0.3)

# 右：不同 lambda 的验证集误差
ax2 = axes[1]
test_mse = []
for a in alphas:
    ridge = Ridge(alpha=a)
    ridge.fit(X_tr_scaled, y_tr_ts.astype(float))
    pred = ridge.predict(X_te_scaled)
    test_mse.append(mean_squared_error(y_te_ts, pred))

best_idx = np.argmin(test_mse)
ax2.plot(np.log10(alphas), test_mse, color='steelblue', lw=2)
ax2.axvline(np.log10(alphas[best_idx]), color='crimson', lw=2, linestyle='--',
            label=f'最优 lambda={alphas[best_idx]:.3f}')
ax2.set_xlabel('log10(lambda)')
ax2.set_ylabel('测试集 MSE')
ax2.set_title('正则化强度 vs 测试集误差\n(lambda 过小→过拟合，过大→欠拟合)', fontsize=11)
ax2.legend(fontsize=10)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# RidgeCV 自动选参
ridge_cv = RidgeCV(alphas=alphas, cv=TimeSeriesSplit(5))
ridge_cv.fit(X_tr_scaled, y_tr_ts.astype(float))
print(f'RidgeCV 选定最优 lambda (alpha): {ridge_cv.alpha_:.4f}')
print(f'对应 log10(alpha): {np.log10(ridge_cv.alpha_):.3f}')

## 9.7 逻辑回归：二分类涨跌预测

逻辑回归对 Sigmoid 函数建模：

$$P(y=1|\mathbf{x}) = \frac{1}{1 + e^{-(\beta_0 + \boldsymbol{\beta}^T \mathbf{x})}}$$

它最大化对数似然，在 scikit-learn 中默认使用 L2 正则化，超参数 `C`越小正则化越强。

我们使用 **Pipeline** 封装标准化与模型，彻底防止数据泄露。

In [ ]:
# Cell 7: 完整 Pipeline 训练与评估

# 正确的时序 Pipeline
pipe_final = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    LogisticRegression(C=0.1, max_iter=1000, random_state=42))
])

pipe_final.fit(X_tr_ts, y_tr_ts)
y_pred    = pipe_final.predict(X_te_ts)
y_prob    = pipe_final.predict_proba(X_te_ts)[:, 1]

acc  = accuracy_score(y_te_ts, y_pred)
auc  = roc_auc_score(y_te_ts, y_prob)

print('=== 逻辑回归 + Pipeline 评估结果 (时序划分) ===')
print(f'准确率 (Accuracy): {acc:.4f}')
print(f'AUC-ROC           : {auc:.4f}')
print(f'基准线 (Majority) : {acc_baseline:.4f}')
print()
print('详细分类报告：')
print(classification_report(y_te_ts, y_pred, target_names=['跌(0)', '涨(1)']))

# 系数解读
coef = pipe_final.named_steps['clf'].coef_[0]
print('逻辑回归系数（标准化后）：')
coef_df = pd.DataFrame({'feature': X_all.columns, 'coef': coef})
coef_df = coef_df.sort_values('coef', ascending=False)
print(coef_df.to_string(index=False))
print('\n系数正 \u2192 该特征增大时预测涨的概率大；系数负→预测跌')

## 9.8 ROC 曲线与 AUC

ROC 曲线展示在不同分类阈值下，模型的真阳率（TPR）与假阳率（FPR）的权衡。

- **AUC = 1**：完美分类器
- **AUC = 0.5**：完全随机猜测（对角线）
- **AUC < 0.5**：预测方向反了

AUC 不依赖特定阈值，是衡量模型排序能力的鸦箌指标。

In [ ]:
# Cell 8: ROC 曲线 —— 逻辑回归 vs 随机平底线

fpr, tpr, _ = roc_curve(y_te_ts, y_prob)

# 随机平底线基准
random_prob = np.full(len(y_te_ts), 0.5)
fpr_rand, tpr_rand, _ = roc_curve(y_te_ts, random_prob)
auc_rand = roc_auc_score(y_te_ts, random_prob)

fig, ax = plt.subplots(figsize=(7, 7))
ax.plot(fpr, tpr, color='steelblue', lw=2.5,
        label=f'逻辑回归 (AUC = {auc:.3f})')
ax.plot([0, 1], [0, 1], color='gray', lw=1.5, linestyle='--',
        label=f'随机猜测 (AUC = {auc_rand:.3f})')
ax.fill_between(fpr, tpr, alpha=0.15, color='steelblue')
ax.set_xlim([0, 1])
ax.set_ylim([0, 1])
ax.set_xlabel('假阳率 (FPR)', fontsize=12)
ax.set_ylabel('真阳率 (TPR)', fontsize=12)
ax.set_title('ROC 曲线\nTECH 涨跌预测（时序划分）', fontsize=13)
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f'逻辑回归 AUC = {auc:.4f}')
print(f'随机基准 AUC = {auc_rand:.4f}')
if auc > 0.55:
    print('结论：模型 AUC > 0.55，具有一定排序能力')
else:
    print('结论：模型 AUC 近其0.5，所选特征对明日涨跌预测力有限')

## 9.9 信息系数 IC 与 RankIC

**信息系数 IC**（量化投资核心指标）：

$$IC_t = \text{Corr}\bigl(\hat{r}_{i,t},\; r_{i,t+1}\bigr)$$

即第 $t$ 期预测收益与第 $t+1$ 期实际收益在截面上的 Pearson 相关系数。

**RankIC**：改用 Spearman 秩相关，对极端専更鲁棒， A 股实践中更常用。

| 指标 | 参考阈値 |
|------|--------|
| IC | |IC| > 0.02 日频有参考价值 |
| ICIR = 均値/标准差 | > 0.5 地基价値稳定 |

我们用 **20 日动量特征**预测 4 只股票明日收益，实抱计算 IC 序列。

In [ ]:
# Cell 9: 信息系数 IC 与 RankIC 计算

# 构造所有股票的 20日动量特征与明日收益
all_feat = pd.DataFrame(index=rets.index)
all_label = pd.DataFrame(index=rets.index)

for tk in TICKERS:
    all_feat[tk]  = rets[tk].rolling(20).sum().shift(1)   # 20日动量延迟1期
    all_label[tk] = rets[tk].shift(-1)                     # 明日收益

# 逐日计算横截面 IC：对 4 只股票计算当日 IC
valid_idx = all_feat.dropna().index.intersection(all_label.dropna().index)
ic_series    = pd.Series(dtype=float, name='IC')
rankic_series = pd.Series(dtype=float, name='RankIC')

for dt in valid_idx:
    f_cross = all_feat.loc[dt].dropna()
    l_cross = all_label.loc[dt].reindex(f_cross.index).dropna()
    f_cross = f_cross.reindex(l_cross.index)
    if len(f_cross) < 3:
        continue
    ic_val, _ = pearsonr(f_cross, l_cross)
    ric_val, _ = spearmanr(f_cross, l_cross)
    ic_series[dt]     = ic_val
    rankic_series[dt] = ric_val

ic_mean    = ic_series.mean()
ic_std     = ic_series.std()
icir       = ic_mean / ic_std if ic_std > 0 else 0
ric_mean   = rankic_series.mean()
ric_std    = rankic_series.std()
ricir      = ric_mean / ric_std if ric_std > 0 else 0

print('=== 20日动量因子 IC 统计 (4只股票截面) ===')
print(f'IC 均値 : {ic_mean:.4f}  \u00b1  {ic_std:.4f}')
print(f'ICIR     : {icir:.4f}')
print(f'RankIC 均値: {ric_mean:.4f}  \u00b1  {ric_std:.4f}')
print(f'RankICIR  : {ricir:.4f}')
print(f'IC>0 比例: {(ic_series>0).mean():.1%}')

# IC 时序图
fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)

axes[0].plot(ic_series.index, ic_series.values, color='steelblue', lw=1, alpha=0.8)
axes[0].axhline(0,        color='black', lw=0.8, linestyle=':')
axes[0].axhline(ic_mean,  color='crimson', lw=2, linestyle='--', label=f'IC 均値={ic_mean:.3f}')
axes[0].fill_between(ic_series.index, ic_series.values, 0,
                     where=ic_series.values > 0, color='steelblue', alpha=0.3)
axes[0].fill_between(ic_series.index, ic_series.values, 0,
                     where=ic_series.values < 0, color='crimson',   alpha=0.3)
axes[0].set_ylabel('IC (Pearson)')
axes[0].set_title('20日动量因子日频 IC 时序', fontsize=12)
axes[0].legend(fontsize=10)

axes[1].plot(rankic_series.index, rankic_series.values, color='darkorange', lw=1, alpha=0.8)
axes[1].axhline(0,         color='black', lw=0.8, linestyle=':')
axes[1].axhline(ric_mean,  color='crimson', lw=2, linestyle='--', label=f'RankIC 均値={ric_mean:.3f}')
axes[1].fill_between(rankic_series.index, rankic_series.values, 0,
                     where=rankic_series.values > 0, color='darkorange', alpha=0.3)
axes[1].fill_between(rankic_series.index, rankic_series.values, 0,
                     where=rankic_series.values < 0, color='crimson',    alpha=0.3)
axes[1].set_ylabel('RankIC (Spearman)')
axes[1].set_title('20日动量因子日频 RankIC 时序', fontsize=12)
axes[1].legend(fontsize=10)
axes[1].set_xlabel('日期')

plt.tight_layout()
plt.show()

## 9.10 多股票对比：全部 4 只股票的涨跌预测能力

对 4 只股票分别训练逻辑回归，对比各股票在时序划分下的准确率与 AUC。

In [ ]:
# Cell 10: 4 只股票涨跌预测对比

results_all = []

for tk in TICKERS:
    r_tk = rets[tk].copy()

    # 构造特征
    f_tk = pd.DataFrame(index=r_tk.index)
    f_tk['ret_1']  = r_tk.shift(1)
    f_tk['ret_5']  = r_tk.rolling(5).sum().shift(1)
    f_tk['ret_20'] = r_tk.rolling(20).sum().shift(1)
    f_tk['vol_20'] = r_tk.rolling(20).std().shift(1)
    ma5_tk  = prices[tk].rolling(5).mean()
    ma20_tk = prices[tk].rolling(20).mean()
    f_tk['ma_signal'] = (ma5_tk / ma20_tk - 1).shift(1)
    f_tk['mkt_ret_5'] = mkt_r.rolling(5).sum().shift(1)

    y_tk = (r_tk.shift(-1) > 0).astype(int)
    y_tk.name = 'label'

    d_tk = f_tk.join(y_tk).dropna()
    X_tk = d_tk[f_tk.columns]
    y_tk = d_tk['label']

    # 时序划分
    sp = int(len(X_tk) * 0.7)
    Xtr, Xte = X_tk.iloc[:sp], X_tk.iloc[sp:]
    ytr, yte = y_tk.iloc[:sp], y_tk.iloc[sp:]

    pipe_tk = Pipeline([
        ('scaler', StandardScaler()),
        ('clf',    LogisticRegression(C=0.1, max_iter=1000, random_state=42))
    ])
    pipe_tk.fit(Xtr, ytr)

    ypred = pipe_tk.predict(Xte)
    yprob = pipe_tk.predict_proba(Xte)[:, 1]

    acc_tk  = accuracy_score(yte, ypred)
    auc_tk  = roc_auc_score(yte, yprob)
    base_tk = max(ytr.mean(), 1 - ytr.mean())

    # TimeSeriesSplit CV
    tscv_tk = TimeSeriesSplit(n_splits=5)
    cv_scores_tk = cross_val_score(pipe_tk, X_tk, y_tk, cv=tscv_tk, scoring='accuracy')

    results_all.append({
        '股票': tk,
        '测试集准确率': round(acc_tk, 4),
        '测试集 AUC': round(auc_tk, 4),
        'CV准确率均': round(cv_scores_tk.mean(), 4),
        'CV准确率std': round(cv_scores_tk.std(), 4),
        '多数类基准': round(base_tk, 4),
        '超出基准': round(acc_tk - base_tk, 4),
    })

df_results = pd.DataFrame(results_all).set_index('股票')
print('=== 4 只股票涨跌预测对比（时序划分）===')
print(df_results.to_string())

# 可视化
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax1 = axes[0]
colors_bar = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
x = np.arange(len(TICKERS))
bars = ax1.bar(x, df_results['测试集准确率'], color=colors_bar, alpha=0.8, edgecolor='white')
for i, tk in enumerate(TICKERS):
    ax1.axhline(df_results.loc[tk, '多数类基准'],
                xmin=(i+0.1)/len(TICKERS), xmax=(i+0.9)/len(TICKERS),
                color='black', lw=2, linestyle='--')
ax1.set_xticks(x); ax1.set_xticklabels(TICKERS)
ax1.set_ylabel('测试集准确率')
ax1.set_title('各股票涨跌预测准确率\n（黑线为多数类基准）', fontsize=11)
ax1.set_ylim(0.4, 0.75)

ax2 = axes[1]
ax2.bar(x, df_results['测试集 AUC'], color=colors_bar, alpha=0.8, edgecolor='white')
ax2.axhline(0.5, color='gray', lw=1.5, linestyle='--', label='AUC=0.5 (随机基准)')
ax2.set_xticks(x); ax2.set_xticklabels(TICKERS)
ax2.set_ylabel('AUC-ROC')
ax2.set_title('各股票涨跌预测 AUC', fontsize=11)
ax2.set_ylim(0.3, 0.75)
ax2.legend()

plt.suptitle('4 只股票涨跌预测能力对比 (时序划分)', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 9.11 值得注意：偏差-方差权衡可视化

通过调节 `C`（正则化强度的倒数），展示训练集与验证集准确率的权衡关系。

In [ ]:
# Cell 11: 偏差-方差权衡可视化

C_values = np.logspace(-3, 2, 30)
train_accs = []
val_accs   = []

for C_val in C_values:
    pipe_bv = Pipeline([
        ('scaler', StandardScaler()),
        ('clf',    LogisticRegression(C=C_val, max_iter=1000, random_state=42))
    ])
    pipe_bv.fit(X_tr_ts, y_tr_ts)
    train_accs.append(accuracy_score(y_tr_ts, pipe_bv.predict(X_tr_ts)))
    val_accs.append(accuracy_score(y_te_ts, pipe_bv.predict(X_te_ts)))

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(np.log10(C_values), train_accs, 'o-', color='steelblue', lw=2,
        markersize=5, label='训练集准确率')
ax.plot(np.log10(C_values), val_accs,   's-', color='crimson',   lw=2,
        markersize=5, label='测试集准确率')

best_C_idx = np.argmax(val_accs)
ax.axvline(np.log10(C_values[best_C_idx]), color='darkorange', lw=2, linestyle='--',
           label=f'最优 C={C_values[best_C_idx]:.3f}')
ax.axhline(acc_baseline, color='gray', lw=1.5, linestyle=':', label=f'基准线={acc_baseline:.3f}')

ax.set_xlabel('log10(C)  (C小\u2192强正则化 / C大\u2192弱正则化)', fontsize=11)
ax.set_ylabel('准确率')
ax.set_title('逻辑回归正则化强度 vs 准确率\n偏差-方差权衡：C过大\u2192训练集过拟合，C过小\u2192欠拟合', fontsize=12)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f'训练集准确率在 C={C_values[-1]:.0f} 时: {train_accs[-1]:.4f}')
print(f'测试集准确率在 C={C_values[-1]:.0f} 时: {val_accs[-1]:.4f}')
print(f'最优 C={C_values[best_C_idx]:.4f} 时测试集准确率: {val_accs[best_C_idx]:.4f}')

## 9.12 小结：监督学习金融实战关键要点

| 环节 | 关键要求 | 常见错误 |
|------|---------|--------|
| 标签构造 | `shift(-1)` 取未来，`shift(1)` 错开特征 | 特征标签时间对不齐 |
| 数据划分 | `TimeSeriesSplit` | 随机 K-Fold 导致前视偏差 |
| 预处理 | `Pipeline` 内封装 Scaler | 先全局标准化再划分 |
| 正则化 | 时序 CV 选 $\lambda$ | 随机 CV 选出虚高 C |
| 评估 | IC/RankIC + AUC | 只看准确率，忽视基准线 |

> **金融机器学习黄金法则**：时间永远单向流动，特征必须严格先于标签；
> 用 `Pipeline` 封装所有预处理；超参数调整也在时序框架内进行。

## 习题参考解答

以下代码对应正文第 9.11 节习题，可直接运行。

In [ ]:
# 习题 9.1：标签构造与无前视偏差验证
print('=' * 55)
print('习题 9.1: 标签构造（BANK 股票）')
print('=' * 55)

r_bank = rets['BANK']

# (a) 明日简单收益率（回归标签）
label_reg = r_bank.shift(-1)
label_reg.name = '明日收益'

# (b) 未来 5 日累计收益是否为正（分类标签）
label_cls_5d = (r_bank.rolling(5).sum().shift(-5) > 0).astype(int)
label_cls_5d.name = '5日上涨'

print('\n(a) 明日收益率（回归标签）前5行：')
print(label_reg.dropna().head())
print('\n(b) 5日涨跌分类标签前5行：')
print(label_cls_5d.dropna().head())
print(f'\n5日上涨占比: {label_cls_5d.mean():.1%}')

# 可视化
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

label_reg.dropna().hist(bins=40, ax=axes[0], color='steelblue', alpha=0.7, edgecolor='white')
axes[0].set_title('(a) BANK 明日收益率分布（回归标签）', fontsize=11)
axes[0].set_xlabel('收益率')
axes[0].set_ylabel('频次')
axes[0].axvline(0, color='crimson', lw=2, linestyle='--')

label_cls_5d.value_counts().sort_index().plot(kind='bar', ax=axes[1],
    color=['crimson', 'steelblue'], alpha=0.8, edgecolor='white', rot=0)
axes[1].set_title('(b) BANK 5日涨跌标签分布', fontsize=11)
axes[1].set_xlabel('标签：0=跌，1=涨')
axes[1].set_ylabel('样本数')

plt.suptitle('习题 9.1: 标签构造与分布', y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

print('\n无前视偏差验证：')
print('  - 明日收益: shift(-1) \u2192 t+1 时刻的实际收益，特征必须使用 shift(1) 及以先的数据')
print('  - 5日涨跌: shift(-5) \u2192 [t+1, t+5] 时窗的未来信息，特征同理延迟')

In [ ]:
# 习题 9.2: KFold vs TimeSeriesSplit 对比（TECH 股票）
print('=' * 55)
print('习题 9.2: KFold vs TimeSeriesSplit 交叉验证（TECH）')
print('=' * 55)

from sklearn.model_selection import KFold

pipe_q2 = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    LogisticRegression(C=0.1, max_iter=1000, random_state=42))
])

# (a) 5 折随机 KFold
kf_q2 = KFold(n_splits=5, shuffle=True, random_state=42)
kf_scores = cross_val_score(pipe_q2, X_all, y_all, cv=kf_q2, scoring='accuracy')

# (b) 5 折 TimeSeriesSplit
ts_q2 = TimeSeriesSplit(n_splits=5)
ts_scores = cross_val_score(pipe_q2, X_all, y_all, cv=ts_q2, scoring='accuracy')

print(f'\n(a) KFold(随机) 均准确率: {kf_scores.mean():.4f} +/- {kf_scores.std():.4f}')
print(f'    每折: {[round(s,4) for s in kf_scores]}')
print(f'\n(b) TimeSeriesSplit 均准确率: {ts_scores.mean():.4f} +/- {ts_scores.std():.4f}')
print(f'    每折: {[round(s,4) for s in ts_scores]}')
print(f'\n(c) 虚高幅度: {kf_scores.mean() - ts_scores.mean():.4f}')
print('''
解释: KFold 的高分来自前视偏差——
  随机打乱后，t=100 的训练样本和 t=99 的验证样本共享了时间临近的信息。
  TimeSeriesSplit 验证集总在训练集之后，不存在时间倒流，更斟实。
  KFold 得分高出的部分在实盘中无法复现，最终根本没有价値。
''')

In [ ]:
# 习题 9.3: 岖回归正则化路径
print('=' * 55)
print('习题 9.3: 岖回归正则化路径图')
print('=' * 55)

lambdas = [0.001, 0.01, 0.1, 1, 10]

scaler_q3 = StandardScaler()
X_tr_q3 = scaler_q3.fit_transform(X_tr_ts)
X_te_q3 = scaler_q3.transform(X_te_ts)

print('\n(a) 正则化路径（系数表）:')
coef_rows = []
for lam in lambdas:
    ridge_q3 = Ridge(alpha=lam)
    ridge_q3.fit(X_tr_q3, y_tr_ts.astype(float))
    row = {'lambda': lam}
    for i, col in enumerate(X_all.columns):
        row[col] = round(ridge_q3.coef_[i], 4)
    coef_rows.append(row)

df_path = pd.DataFrame(coef_rows).set_index('lambda')
print(df_path.to_string())

print('\n(b) 观察: lambda 增大，各系数的绝对値均向零收缩，但不会精确为零（L2）。')

# (c) RidgeCV 自动选参
alphas_q3 = np.logspace(-3, 2, 50)
ridge_cv_q3 = RidgeCV(alphas=alphas_q3, cv=TimeSeriesSplit(5))
ridge_cv_q3.fit(X_tr_q3, y_tr_ts.astype(float))
print(f'\n(c) RidgeCV 最优 lambda: {ridge_cv_q3.alpha_:.4f}')

# 绘路径图
alphas_plot = np.logspace(-3, 2, 60)
coef_path_q3 = []
for a in alphas_plot:
    r = Ridge(alpha=a).fit(X_tr_q3, y_tr_ts.astype(float))
    coef_path_q3.append(r.coef_)
coef_path_q3 = np.array(coef_path_q3)

fig, ax = plt.subplots(figsize=(10, 6))
for i, col in enumerate(X_all.columns):
    ax.plot(np.log10(alphas_plot), coef_path_q3[:, i], lw=2, label=col)
ax.axvline(np.log10(ridge_cv_q3.alpha_), color='black', lw=2, linestyle='--',
           label=f'RidgeCV 最优 lambda={ridge_cv_q3.alpha_:.3f}')
ax.axhline(0, color='gray', lw=0.8)
ax.set_xlabel('log10(lambda)')
ax.set_ylabel('系数大小')
ax.set_title('岖回归正则化路径图', fontsize=12)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# 习题 9.4: IC 计算与 ICIR 分析
print('=' * 55)
print('习题 9.4: 20日动量师子 IC/RankIC 分析')
print('=' * 55)

# 重用之前计算的 ic_series, rankic_series
print(f'\n(a) IC 序列长度: {len(ic_series)}，有效日期: {ic_series.index[0].date()} \u2013 {ic_series.index[-1].date()}')
print(f'    IC 均値: {ic_series.mean():.4f}， IC 标准差: {ic_series.std():.4f}')
print(f'\n(b) ICIR  = IC均値 / IC标准差 = {icir:.4f}')
print(f'    RankICIR = {ricir:.4f}')

# 滚动均値
ic_rolling_mean = ic_series.rolling(60).mean()
ric_rolling_mean = rankic_series.rolling(60).mean()

fig, ax = plt.subplots(figsize=(13, 5))
ax.bar(ic_series.index, ic_series.values, color='lightblue', alpha=0.5, label='IC 日度')
ax.plot(ic_rolling_mean.index, ic_rolling_mean.values, color='steelblue', lw=2,
        label='IC 60日滚动均値')
ax.plot(ric_rolling_mean.index, ric_rolling_mean.values, color='darkorange', lw=2,
        linestyle='--', label='RankIC 60日滚动均値')
ax.axhline(0, color='black', lw=0.8)
ax.axhline(ic_series.mean(), color='crimson', lw=1.5, linestyle=':', label=f'全期 IC均値={ic_series.mean():.3f}')
ax.set_xlabel('日期')
ax.set_ylabel('IC / RankIC')
ax.set_title('习题 9.4: 20日动量师子 IC 时序与滚动均値', fontsize=12)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f'\n(c) IC>0 占比: {(ic_series>0).mean():.1%}')
if abs(ic_series.mean()) < 0.02:
    print('    IC 均値绝对値 < 0.02, 该师子预测力较弱（正常，内置合成数据）')
else:
    print('    IC 均値 |IC| >= 0.02, 师子具有一定预测价値')

In [ ]:
# 习题 9.5: ROC 曲线对比分析
print('=' * 55)
print('习题 9.5: ROC 曲线与 AUC 分析')
print('=' * 55)

# 逻辑回归 (Pipeline 已在上面训练完毕)
y_prob_lr = pipe_final.predict_proba(X_te_ts)[:, 1]
fpr_lr, tpr_lr, _ = roc_curve(y_te_ts, y_prob_lr)
auc_lr = roc_auc_score(y_te_ts, y_prob_lr)

# 随机猜测基准
np.random.seed(42)
y_prob_rand = np.random.uniform(0, 1, len(y_te_ts))
fpr_rand2, tpr_rand2, _ = roc_curve(y_te_ts, y_prob_rand)
auc_rand2 = roc_auc_score(y_te_ts, y_prob_rand)

print(f'\n(a) 逻辑回归 AUC: {auc_lr:.4f}')
print(f'(b) 随机猜测 AUC: {auc_rand2:.4f} (应接近0.5)')
print(f'(c) AUC 差値: {auc_lr - auc_rand2:.4f}')

fig, ax = plt.subplots(figsize=(7, 7))
ax.plot(fpr_lr,    tpr_lr,    color='steelblue', lw=2.5,
        label=f'逻辑回归 (AUC={auc_lr:.3f})')
ax.plot(fpr_rand2, tpr_rand2, color='crimson',   lw=1.5, linestyle='-.',
        label=f'随机猜测 (AUC={auc_rand2:.3f})')
ax.plot([0, 1], [0, 1], color='gray', lw=1, linestyle='--', label='对角线 (AUC=0.5)')
ax.fill_between(fpr_lr, tpr_lr, alpha=0.15, color='steelblue')
ax.set_xlabel('假阳率 FPR', fontsize=12)
ax.set_ylabel('真阳率 TPR', fontsize=12)
ax.set_title('习题 9.5: ROC 曲线对比', fontsize=13)
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print('''
AUC = 0.5 的经济含义：
  - AUC 衡量的是“对于随机抽取的涨天样本和跌天样本，
    模型正确判断涨天得分更高的概率”。
  - AUC=0.5 意味着这个概率只有5O%，完全没有判别能力，
    等价于随机猎头。
  - 在金融中， AUC > 0.55 可用，AUC > 0.60 属于较强信号。
''')